<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-03/AI_Modul_3.4_Praktikum_Variabel_dan_Tipe_Data.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 3.4: Praktikum Variabel dan Tipe Data
### Model Memori Referensi Objek PyObject, Paradigma Dynamic & Strong Typing, Klasifikasi Tipe Data Skalar, Audit Presisi Numerik IEEE 754, dan Pengolahan Telemetri Sensorik Perkebunan

---

> **Diktat Terkait:** [AI_Modul_3.4_Variabel_dan_Tipe_Data.md](../../docs/part-03/AI_Modul_3.4_Variabel_dan_Tipe_Data.md)  
> **Outputs:** Skrip inspeksi jejak memori CPython (`id()`, `sys.getrefcount()`, `sys.getsizeof()`), modul sanitasi telemetri sensorik multivariat defensif, serta pustaka pembanding presisi numerik IEEE 754.  
> **Outcomes:** Mahasiswa menguasai konsep *reference binding*, memahami imutabilitas objek skalar, terampil menangani data hilang (*missing values / None*), dan mampu mengatasi galat pembulatan fraksi desimal biner.  
> **Impacts:** Terjaminnya integritas komputasi metrik agrikultur cerdas, optimalisasi alokasi memori RAM pada *Edge AI IoT*, dan kesiapan menuju manipulasi tensor machine learning.
>
> **Tujuan Praktikum:**  
> 1. Memverifikasi alamat memori dan pencacah referensi `PyObject` di Heap.  
> 2. Menguji fenomena *Small Integer Interning* dan keterbatasan fraksi biner IEEE 754 (`0.1 + 0.2 != 0.3`).  
> 3. Mengaudit jejak alokasi byte fisik tipe skalar menggunakan `sys.getsizeof()`.  
> 4. Menjalankan pipeline sanitasi data telemetri iklim mikro perkebunan sawit (`SanitizerTelemetriKebun`).  
> 5. Menyelesaikan 3 tantangan: sanitasi parameter tanah, audit presisi pecahan apung, dan pelacak jejak memori rekursif.

## 1. Persiapan Environment & Pemuatan Library
Jalankan sel berikut untuk memuat pustaka inspeksi memori CPython dan modul visualisasi:

In [ ]:
import sys
import math
import decimal
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

# Modul numerik & visualisasi
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Konfigurasi visualisasi grafik
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

print("[OK] Seluruh pustaka dasar dan analitika berhasil diinisialisasi.")
print(f"[INFO] Arsitektur Pointer CPython: {sys.maxsize.bit_length() + 1}-Bit")

## 2. Eksplorasi Arsitektur Memori PyObject: Identitas, Tipe, & Referensi
Membuktikan secara empiris bahwa variabel Python adalah label referensi (*pointer binding*) menuju `PyObject` di memori Heap, bukan wadah penyimpan nilai fisik.

In [ ]:
# Demonstrasi Reference Binding
suhu_awal = 31.75
bacaan_sensor = suhu_awal  # Mengikat label kedua ke objek float yang sama

print("=== ANALISIS ARSITEKTUR MEMORI PYOBJECT ===")
print(f"Nilai suhu_awal     : {suhu_awal} | Alamat Memori (id): {hex(id(suhu_awal))}")
print(f"Nilai bacaan_sensor : {bacaan_sensor} | Alamat Memori (id): {hex(id(bacaan_sensor))}")
print(f"Apakah Menunjuk ke Objek yang Sama? (is) : {suhu_awal is bacaan_sensor}")
print(f"Apakah Nilainya Sama? (==)              : {suhu_awal == bacaan_sensor}")

# Pembuktian Imutabilitas Skalar: Mengubah nilai suhu_awal
suhu_awal = suhu_awal + 0.5
print("\nSetelah dilakukan penambahan (suhu_awal = suhu_awal + 0.5):")
print(f"suhu_awal baru      : {suhu_awal} | Alamat Memori Baru : {hex(id(suhu_awal))}")
print(f"bacaan_sensor lama  : {bacaan_sensor} | Alamat Memori Tetap: {hex(id(bacaan_sensor))}")
print(f"Apakah Masih Menunjuk Objek Sama? (is)  : {suhu_awal is bacaan_sensor}")

## 3. Investigasi Integer Interning & Presisi Pecahan Apung IEEE 754
Menguji perilaku caching bilangan bulat kecil (`-5` s/d `256`) serta mengaudit anomali representasi biner pecahan desimal (`0.1 + 0.2 != 0.3`).

In [ ]:
# 1. Eksperimen Small Integer Interning Cache
int_a = 256
int_b = 256
print("=== 1. SMALL INTEGER INTERNING CACHE ===")
print(f"int_a = {int_a} (id: {id(int_a)}) | int_b = {int_b} (id: {id(int_b)})")
print(f"int_a is int_b (Rentang Cache <= 256) : {int_a is int_b}")

# 2. Eksperimen Keterbatasan Presisi Pecahan Biner IEEE 754
f1 = 0.1
f2 = 0.2
f_hasil = f1 + f2

print("\n=== 2. AUDIT PRESISI FRAKSI DESIMAL BINER IEEE 754 ===")
print(f"0.1 + 0.2 Hasil Standar   : {f_hasil}")
print(f"Format Presisi Tinggi (55): {f_hasil:.55f}")
print(f"Evaluasi 0.1 + 0.2 == 0.3 : {f_hasil == 0.3} (Anomali Biner)")
print(f"Evaluasi via math.isclose : {math.isclose(f_hasil, 0.3, rel_tol=1e-9)} (Solusi Toleransi Sains)")

# 3. Solusi Presisi Mutlak Menggunakan decimal.Decimal
d1 = decimal.Decimal("0.1")
d2 = decimal.Decimal("0.2")
d_hasil = d1 + d2
print(f"Solusi Finansial (Decimal): {d1} + {d2} == {d_hasil} -> {d_hasil == decimal.Decimal('0.3')}")

## 4. Audit Jejak Alokasi Memori Fisik Skalar (`sys.getsizeof`)
Membandingkan konsumsi byte memori antara tipe data skalar Python untuk memahami beban alokasi RAM pada perangkat tepi (*Edge AI*).

In [ ]:
tipe_sampel = [
    ("Integer 0", 0),
    ("Integer Kecil (100)", 100),
    ("Integer Raksasa (2**100)", 2**100),
    ("Float (31.75)", 31.75),
    ("Boolean (True)", True),
    ("NoneType (None)", None),
    ("String Kosong ('')", ""),
    ("String Kode ('BLOK-A01')", "BLOK-A01"),
]

nama_labels = []
ukuran_bytes = []

print("=== KONSUMSI MEMORI FISIK STRUKTUR PYOBJECT (64-BIT) ===")
print(f"{'Tipe Data':<28} | {'Ukuran Fisik (Byte)':<20} | {'Tipe Kelas'}")
print("-" * 68)

for label, obj in tipe_sampel:
    sz = sys.getsizeof(obj)
    nama_labels.append(label)
    ukuran_bytes.append(sz)
    print(f"{label:<28} | {sz:<20} | {type(obj)}")

# Visualisasi Grafik Jejak Memori
fig, ax = plt.subplots(figsize=(8.5, 4.5))
bars = ax.barh(nama_labels, ukuran_bytes, color='steelblue', edgecolor='black', height=0.6)
ax.set_xlabel('Ukuran Alokasi RAM (Byte)', fontweight='bold')
ax.set_title('Jejak Alokasi Memori Fisik Objek Skalar Python (sys.getsizeof)', fontweight='bold', fontsize=11)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.8, bar.get_y() + bar.get_height()/2., f'{w} B', va='center', fontweight='bold', fontsize=8.5)

plt.tight_layout()
plt.savefig('docs/assets/jejak_memori_tipe_skalar.png', dpi=150)
print("\n[INFO] Grafik alokasi memori berhasil disimpan ke 'docs/assets/jejak_memori_tipe_skalar.png'.")

## 5. Implementasi Terpadu Pipeline Industri: `SanitizerTelemetriKebun`
Mengeksekusi sistem pembersihan dan standarisasi tipe data telemetri agroklimat perkebunan sawit terintegrasi.

In [ ]:
@dataclass(frozen=True)
class RekamanTelemetriMentah:
    id_stasiun: Any
    suhu_udara: Any
    kelembaban_tanah: Any
    curah_hujan: Any
    status_pompa: Any

@dataclass(frozen=True)
class TelemetriTersanitasi:
    id_stasiun: str
    suhu_udara_c: float
    kelembaban_tanah_pct: float
    curah_hujan_mm: float
    status_pompa_aktif: bool
    memiliki_imputasi: bool

class SanitizerTelemetriKebun:
    SUHU_DEFAULT_C: float = 27.5
    KELEMBABAN_DEFAULT_PCT: float = 65.0

    @classmethod
    def konversi_ke_float_aman(cls, nilai_mentah: Any, nilai_baku: float) -> Tuple[float, bool]:
        if nilai_mentah is None:
            return nilai_baku, True
        if isinstance(nilai_mentah, (int, float)):
            return float(nilai_mentah), False
        if isinstance(nilai_mentah, str):
            teks = nilai_mentah.strip()
            if not teks or teks.upper() in ["NULL", "NAN", "NA", "ERR"]:
                return nilai_baku, True
            try:
                return float(teks), False
            except ValueError:
                return nilai_baku, True
        return nilai_baku, True

    @classmethod
    def konversi_ke_bool_aman(cls, nilai_mentah: Any) -> bool:
        if isinstance(nilai_mentah, bool):
            return nilai_mentah
        if isinstance(nilai_mentah, (int, float)):
            return bool(nilai_mentah != 0)
        if isinstance(nilai_mentah, str):
            teks = nilai_mentah.strip().upper()
            return teks in ["TRUE", "1", "ON", "AKTIF", "YES"]
        return False

    def proses_batch_telemetri(self, kumpulan_data: List[RekamanTelemetriMentah]) -> List[TelemetriTersanitasi]:
        hasil = []
        for item in kumpulan_data:
            id_str = str(item.id_stasiun).strip().upper() if item.id_stasiun else "STASIUN_ANONIM"
            s_f, imp_s = self.konversi_ke_float_aman(item.suhu_udara, self.SUHU_DEFAULT_C)
            rh_f, imp_rh = self.konversi_ke_float_aman(item.kelembaban_tanah, self.KELEMBABAN_DEFAULT_PCT)
            h_f, imp_h = self.konversi_ke_float_aman(item.curah_hujan, 0.0)
            p_b = self.konversi_ke_bool_aman(item.status_pompa)
            imputasi = imp_s or imp_rh or imp_h
            
            hasil.append(TelemetriTersanitasi(
                id_stasiun=id_str,
                suhu_udara_c=round(s_f, 2),
                kelembaban_tanah_pct=round(rh_f, 2),
                curah_hujan_mm=round(h_f, 2),
                status_pompa_aktif=p_b,
                memiliki_imputasi=imputasi
            ))
        return hasil

# Eksekusi Sanitasi Data Lapang
aliran_mentah = [
    RekamanTelemetriMentah("WS-01", "29.40", "62.5", "0.0", "ON"),
    RekamanTelemetriMentah("WS-01", 31.2, None, "12.5", 1),
    RekamanTelemetriMentah("WS-02", "ERR", "78.0", "0.0", "False"),
    RekamanTelemetriMentah(None, "27.8", " 55.4 ", None, "AKTIF"),
    RekamanTelemetriMentah("WS-03", 26.5, 88.0, 45.0, False),
]

sanitizer = SanitizerTelemetriKebun()
laporan_bersih = sanitizer.proses_batch_telemetri(aliran_mentah)

print("=== LAPORAN HASIL SANITASI TELEMETRI KEBUN SAWIT ===")
for d in laporan_bersih:
    st = "[TERIMPUTASI]" if d.memiliki_imputasi else "[ASLI]"
    p_status = "HIDUP" if d.status_pompa_aktif else "MATI"
    print(f"{d.id_stasiun:<14}: Suhu: {d.suhu_udara_c:<5}°C | RH: {d.kelembaban_tanah_pct:<5}% | Hujan: {d.curah_hujan_mm:<5} mm | Pompa: {p_status:<5} | {st}")

## 6. Solusi Lengkap Tantangan Pemrograman Berjenjang
Berikut adalah implementasi komprehensif dari ketiga tantangan mandiri yang terdapat pada diktat perkuliahan:

### Tantangan 1 (Tingkat Dasar): Sanitasi Parameter Sensor Tanah dengan Imputasi Rerata
Membersihkan deret data kelembaban tanah mentah dari string dan nilai null menjadi float tervalidasi.

In [ ]:
def sanitasi_telemetri_sensor(daftar_mentah: List[Any], nilai_fallback: float = 45.0) -> List[float]:
    """
    Menyaring dan mengonversi daftar pembacaan sensor mentah menjadi list float bersih.
    """
    hasil_float: List[float] = []
    for item in daftar_mentah:
        if item is None:
            hasil_float.append(nilai_fallback)
            continue
        if isinstance(item, (int, float)):
            hasil_float.append(float(item))
            continue
        if isinstance(item, str):
            teks = item.strip()
            try:
                hasil_float.append(float(teks))
            except ValueError:
                hasil_float.append(nilai_fallback)
        else:
            hasil_float.append(nilai_fallback)
    return hasil_float

# Pengujian Tantangan 1
data_uji_t1 = ["45.2", " 50.1 ", None, "KORUP", "48.5", "NULL"]
hasil_t1 = sanitasi_telemetri_sensor(data_uji_t1, nilai_fallback=45.0)

print("=== TANTANGAN 1: SANITASI PARAMETER TANAH ===")
print(f"Data Mentah Sensor : {data_uji_t1}")
print(f"Data Hasil Bersih  : {hasil_t1}")

### Tantangan 2 (Tingkat Menengah): Evaluator Presisi Pecahan Apung IEEE 754
Membandingkan kesetaraan dua bilangan floating-point secara langsung versus menggunakan toleransi relatif `math.isclose()`.

In [ ]:
def audit_presisi_apung(nilai_a: float, nilai_b: float, toleransi: float = 1e-6) -> Dict[str, Any]:
    """
    Mengevaluasi kesetaraan pecahan apung dengan memperhitungkan galat roundoff IEEE 754.
    """
    sama_langsung = (nilai_a == nilai_b)
    sama_toleransi = math.isclose(nilai_a, nilai_b, abs_tol=toleransi)
    delta_residu = abs(nilai_a - nilai_b)
    
    return {
        "nilai_a": nilai_a,
        "nilai_b": nilai_b,
        "kesetaraan_langsung_sama": sama_langsung,
        "kesetaraan_toleransi_isclose": sama_toleransi,
        "delta_residu_absolut": delta_residu,
        "status_analisis": "Identik Eksak" if sama_langsung else ("Setara Toleransi" if sama_toleransi else "Berbeda Signifikan")
    }

# Pengujian Tantangan 2
akumulasi_tbs_1 = 0.1 + 0.1 + 0.1
akumulasi_tbs_2 = 0.3
laporan_t2 = audit_presisi_apung(akumulasi_tbs_1, akumulasi_tbs_2)

print("=== TANTANGAN 2: AUDIT PRESISI PECAHAN APUNG ===")
print(f"Nilai Akumulasi 1 : {laporan_t2['nilai_a']:.20f}")
print(f"Nilai Akumulasi 2 : {laporan_t2['nilai_b']:.20f}")
print(f"Evaluasi Langsung (==)      : {laporan_t2['kesetaraan_langsung_sama']}")
print(f"Evaluasi Toleransi (isclose): {laporan_t2['kesetaraan_toleransi_isclose']}")
print(f"Residu Pembulatan Biner     : {laporan_t2['delta_residu_absolut']:.2e}")
print(f"Status Hasil Akhir          : {laporan_t2['status_analisis']}")

### Tantangan 3 (Tingkat Mahir): Pelacak Jejak Memori Fisik & Siklus Hidup Objek Sensor
Membangun kelas `MemoryFootprintTracker` untuk menghitung konsumsi memori rekursif dan menganalisis reference count.

In [ ]:
class MemoryFootprintTracker:
    """
    Alat pelacak alokasi byte memori fisik struktur data dan inspeksi siklus hidup objek.
    """
    @staticmethod
    def hitung_total_memori(koleksi_data: Any) -> int:
        """Menghitung total byte alokasi RAM secara rekursif."""
        total = sys.getsizeof(koleksi_data)
        if isinstance(koleksi_data, (list, tuple, set)):
            for elemen in koleksi_data:
                total += MemoryFootprintTracker.hitung_total_memori(elemen)
        elif isinstance(koleksi_data, dict):
            for k, v in koleksi_data.items():
                total += MemoryFootprintTracker.hitung_total_memori(k)
                total += MemoryFootprintTracker.hitung_total_memori(v)
        return total

    @staticmethod
    def inspeksi_objek(nama_label: str, target_objek: Any) -> Dict[str, Any]:
        """Mengaudit alamat memori, tipe data, dan pencacah referensi."""
        # getrefcount mengembalikan +1 karena target_objek dilewatkan sebagai argumen fungsi
        ref_count = sys.getrefcount(target_objek) - 1
        return {
            "label": nama_label,
            "alamat_memori": hex(id(target_objek)),
            "tipe_kelas": str(type(target_objek)),
            "ukuran_byte": sys.getsizeof(target_objek),
            "pencacah_referensi": ref_count
        }

# Pengujian Tantangan 3
tracker = MemoryFootprintTracker()
data_telemetri_nested = {
    "blok": "BLOK-A01",
    "telemetri": [28.5, 30.1, 29.8],
    "status": {"aktif": True, "kode": 101}
}

total_ram = tracker.hitung_total_memori(data_telemetri_nested)
audit_obj = tracker.inspeksi_objek("Data Telemetri Nested", data_telemetri_nested)

print("=== TANTANGAN 3: AUDITOR MEMORI REKURSIF ===")
print(f"Alamat Memori (Heap) : {audit_obj['alamat_memori']}")
print(f"Ukuran Dictionary Luar : {audit_obj['ukuran_byte']} byte")
print(f"Total RAM Rekursif     : {total_ram} byte (Termasuk seluruh sub-elemen)")
print(f"Pencacah Referensi     : {audit_obj['pencacah_referensi']} referensi aktif")